In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`
┌ Warning: The active manifest file is an older format with no julia version entry. Dependencies may have been resolved with a different julia version.
└ @ nothing /Users/Josephine/.julia/dev/drag-formation/Manifest.toml:0


In [2]:
using LinearAlgebra
using SparseArrays
using Convex
using GLPK, MathOptInterface, JuMP
using Plots
using MosekTools
using SatelliteToolbox
using Statistics
using JLD2
# plotlyjs()

pgfplotsx()

Plots.PGFPlotsXBackend()

In [3]:
### Constants
const μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
# const J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)const
const J2 = 1082.63*10^-6 #Earth J2 gravity coefficient (dimensionless) 
const J3 = -2.52*10^-6 #Earth J3 gravity coefficient (dimensionless)
const J4 = -1.61*10^-6 #Earth J4 gravity coefficient (dimensionless)
const J5 = -0.15*10^-6 #Earth J5 gravity coefficient (dimensionless)
const J6 =  0.57*10^-6 #Earth J6 gravity coefficient (dimensionless)
const Re_mks = 6378.137 #Radius of the Earth (km)
const Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

const Cd = 2.2 #Drag coefficient (unitless)
const AoM_mks = (2*(0.2*0.15) + (0.1*0.15))*10^-6/1.504; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 1.504 kg
const AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
# const AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
# const AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [4]:
#Orbital Parameters (roughly ISS)
const alt = 550#440
const a_mks = Re_mks + alt #semi-major axis (km)
const ecc = 0.005
const inc = 98 #inclination (deg)
const n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
const P_mks = 2*pi/n_mks; #orbital period (sec) 1.54 hour 5569.440609858254 sec


In [5]:
#Normalize units
const a = 1.0 #distance unit is semi-major axis
const P = 1 #time unit is orbital period
const μ = (2*pi/P)^2 * a^3 #μ_mks*(P_mks^2/a_mks^3)
const n = sqrt(μ/a^3)
const Re = Re_mks/a_mks
const Ωe = Ωe_mks*P_mks;

const AoM = AoM_mks/(a_mks^2)
const AoM0 = AoM0_mks/(a_mks^2);


In [6]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    θ = atan(x[2],x[1]) #azimuth
    ϕ = atan(sqrt(x[1]^2+x[2]^2),x[3]) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_θaΩ(x)
    r = x[1:3]
    v = x[4:6]
        
    E = 0.5*v'*v + potential(r)
    
    a = -μ/(2*E)
    
    i_x = [1,0,0]
    i_y = [0,1,0]
    i_z = [0,0,1]
    
    h = cross(r,v)
    Ω = atan(h[2]/norm(h[1:2]), h[1]/norm(h[1:2]))
    
    n = cross(i_z,h)/norm(cross(i_z,h))
    if dot(r,i_z) > 0
        θ = acos(dot(n,(r/norm(r))))
    else
        θ = 2*pi- acos(dot(n,(r/norm(r))))
    end
    return [θ; a; Ω]
end

function cartesian_to_ΔθaΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(r0)
    E1 = 0.5*v1'*v1 + potential(r1)
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    
#     oe0 = cartesian_to_oe(x0)
#     oe1 = cartesian_to_oe(x1)

    
#     Δθ = (oe1[5] - oe0[5]) + (oe1[6] - oe0[6])
#     if Δθ < 0
#         Δθ += 2*pi
#     end
#     ΔΩ =  oe1[4] - oe0[4]
#     println("oe1[6] ", oe1[6], " oe0[6] ", oe0[6], "oe1[5] ", oe1[5], " oe0[5] ", oe0[5], " oe1[4] ", oe1[4], " oe0[4] ", oe0[4])
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))

#     println("Δθ cost ", Δθ," Δθ tool ",Δθ_tool," Δω tool ",oe1[5]-oe0[5] )
    h0 = cross(r0,v0)
    K = [0, 0, 1]
    N0 = cross(K,h0)
    if N0[2] >= 0
        Ω0 = acos(N0[1]/norm(N0))
    else
        Ω0 = 2*pi - acos(N0[1]/norm(N0))
    end
#     Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
#     Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    N1 = cross(K,h1)
    if N1[2] >= 0
        Ω1 = acos(N1[1]/norm(N1))
    else
        Ω1 = 2*pi - acos(N1[1]/norm(N1))
    end
    ΔΩ = Ω1-Ω0
    
    return [Δθ; Δa; ΔΩ]
end

function cartesian_to_oe(x)
    r = x[1:3]*10^3
    v = x[4:6]*10^3
    oe = rv_to_kepler(r,v)
    a = oe.a
    e = oe.e
    i = oe.i
    Ω = oe.Ω
    ω = oe.ω
    θ = oe.f
    return [a,e,i,Ω,ω,θ]
end 
function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [7]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [8]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1
function kvalues(a)
    n = sqrt(μ/a^3)
    Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2#-15/a_mks
    ȧ = 2*(a^2/μ)*(a)*n*Fd
    k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
    k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
    k3 = ȧ #dȧ/d(Drag) - 3*sqrt(μ*a)*J2*Re*Re/a^3*sind(inc)^2 
    k4 = k1/k2
    return [k1, k2, k3, k4]
end

kvalues (generic function with 1 method)

In [9]:
#Discrete-time model
num_sat = 3 # number of couple of satellites 
const h = 1 #time step

function create_continuous(a, num_sat)
    #Discrete-time model
#     k1, k2, k3, k4 = kvalues(a)
    A_bar = zeros(3*num_sat,3*num_sat) 
    B_bar = zeros(3*num_sat,num_sat+1) 

    # B_bar[1:3,1:2]=B0
    for k =0:num_sat-1
        k1, k2, k3, k4 = kvalues(a[k+1])
        A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = [0 k2 0; 0 0 0; 0 k1 0]
        B_bar[3*k+2,1] = -k3
        B_bar[3*k+2,2+k] = k3
    end
    return A_bar, B_bar 
end

function c2d(a,dt,num_sat)
    # discretize a continuous time linear system 
    # convert xdot = Ax + Bu 
    # into x_{k+1} = Ad x_k + Bd x_k 
    # where we have dt between k and k +1 
    
    A,B = create_continuous(a, num_sat)
    nx,nu = size(B)
    
    H = exp(dt*[A B;zeros(nu, nx + nu)])
    Ad = H[1:nx, 1:nx]
    Bd = H[1:nx, (nx + 1):(nx + nu)]
    return Ad, Bd 
end

a_vec = a*ones(num_sat)

Ad, Bd = c2d(a_vec, h, num_sat);


In [10]:
function sat_dynamics_rk4(x,u,h)
    f1 = satellite_dynamics(x,u)
    f2 = satellite_dynamics(x + 0.5*h*f1,u)
    f3 = satellite_dynamics(x + 0.5*h*f2, u)
    f4 = satellite_dynamics(x + h*f3, u)
    return x + (h/6.0)*(f1 + 2*f2 +2*f3 +f4)
end

sat_dynamics_rk4 (generic function with 1 method)

In [11]:
### Full Orbital Dynamics with J2 and Drag
function potential(s)
    #Input is position in spherical coordinates
#     r = s[1] #range
#     ϕ = s[2] #azimuth
#     θ = s[3] #elevation measured up from equatorial plane
    
    #Input is position in cartesian coordinates
    r = norm(s)
    sin_ϕ = s[3]/r
    ϕ = atan(sqrt(s[1]^2+s[2]^2)/s[3]) #azimuth
    
    x = sin_ϕ#cos(ϕ)#
    
    U_J2 = μ/r*J2/2*(Re/r)^2*(3*x^2 - 1.0)
    U_J3 = μ/r*J3/2*(Re/r)^3*(5*x^3 - 3*x)
    U_J4 = μ/r*J4/8*(Re/r)^4*(35*x^4 -30*x^2 + 3)
    U_J5 = μ/r*J5/8*(Re/r)^5*(63*x^5 - 70*x^3 + 15*x)
    U_J6 = μ/r*J6/16*(Re/r)^6*(231*x^6 - 315*x^4 + 105*x^2 - 5)
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + U_J2 + U_J3 + U_J4 + U_J5 + U_J6  #(0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

# function to evaluate J and keplerian gravitational acceleration
function gravitational_acceleration(s)
    x = s[1]
    y = s[2]
    z = s[3]
    r = sqrt(s[1]^2+s[2]^2+s[3]^2)
    
    sinϕ = z/r
    g_kp = - μ_mks*s/(r^3)
    g_j2 = -3/2*J2*(μ_mks/r^2)*(Re_mks^2/r^2)*[
        (1-5*sinϕ^2)*x/r;
        (1-5*sinϕ^2)*y/r;
        (3-5*sinϕ^2)*z/r]
    
    g_j3 = -1/2*J3*(μ_mks/r^2)*(Re_mks/r)^3*[
        5*(7*sinϕ^3 - 3*sinϕ)*x/r;
        5*(7*sinϕ^3 - 3*sinϕ)*y/r;
        3*(10*sinϕ^2 - 35/3*sinϕ^4-1)];
    
    g_j4 = -5/8*J4*(μ_mks/r^2)*(Re_mks/r)^4*[
        (3-42*sinϕ^2 + 63*sinϕ^4)*x/r;
        (3-42*sinϕ^2 + 63*sinϕ^4)*y/r;
       -(15-70*sinϕ^2 + 63*sinϕ^4)*z/r];
    
    g_j5 = -1/8*J5*(μ_mks/r^2)*(Re_mks/r)^5*[
        3*(35*sinϕ - 210*sinϕ^3 + 231*sinϕ^5)*x/r;
        3*(35*sinϕ - 210*sinϕ^3 + 231*sinϕ^5)*y/r;
       (15-315*sinϕ^2 + 945*sinϕ^4 - 693*sinϕ^6)];
    
    g_j6 = 1/16*J6*(μ_mks/r^2)*(Re_mks/r)^6*[
        (35 - 945*sinϕ^2 + 3465*sinϕ^4 - 3003*sinϕ^6)*x/r;
        (35 - 945*sinϕ^2 + 3465*sinϕ^4 - 3003*sinϕ^6)*y/r;
        (-315 + 2205*sinϕ^2 - 4851*sinϕ^4 + 3003*sinϕ^6)*z/r];
        
    g = g_kp + g_j2 + g_j3 + g_j4 + g_j5 + g_j6
    return g
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r1 = x[1:3]
    v1 = x[4:6]
    r2 = x[7:9]
    v2 = x[10:12]
    u1 = u[1]
    u2 = u[2]

    
    pos1_x = r1[1]
    pos1_y = r1[2]
    pos1_z = r1[3]
    r1_norm = sqrt(r1[1]^2+r1[2]^2+r1[3]^2)
    
    pos2_x = r2[1]
    pos2_y = r2[2]
    pos2_z = r2[3]
    r2_norm = sqrt(r2[1]^2+r2[2]^2+r2[3]^2)
    
    
    g1 = gravitational_acceleration(r1)
    g2 = gravitational_acceleration(r2)
    
    v1_rel = v1 - cross([0;0;Ωe_mks],r1) #Assume atmosphere co-rotates with the Earth
    ρ1 = density_mks(r1_norm - Re_mks)
    Fdrag1 = -0.5*(u1*AoM_mks)*Cd*ρ1*(norm(v1_rel))*v1_rel
    
    v2_rel = v2 - cross([0;0;Ωe_mks],r2) #Assume atmosphere co-rotates with the Earth
    ρ2 = density_mks(r2_norm - Re_mks)
    Fdrag2 = -0.5*(u2*AoM_mks)*Cd*ρ2*(norm(v2_rel))*v2_rel

    v̇1 = g1 + Fdrag1#- μ_mks*r1/(r1_norm^3) + Fdrag1 #g1
    v̇2 = g2 + Fdrag2#- μ_mks*r2/(r2_norm^3) + Fdrag2 #g2 
    return [v1; v̇1; v2; v̇2]
end


satellite_dynamics (generic function with 1 method)

In [12]:
#Initial and terminal constraints and #Inequality constraints
params = (
    l_12 = 0.03, # number of revolutions
    l_13 = 6.0,
    l_14 = 6.03,
    l_15 = 4.0,
    l_16 = 5.0,
    Δx0 = zeros(size(Ad,1)),
    Δaf = 0, #end at the same altitude
    u_min = 0.2,
    u_max = 1.,
    Δa_min = -15/a_mks,#/num_sat,
    Δa_max = 15/a_mks)#/num_sat) 

k4 = kvalues(a)[4]
println(k4)
params = (; params..., Δθf = 2*pi*[params.l_12,params.l_13,params.l_14,params.l_15,params.l_16],
)

params = (; params..., ΔΩf = k4*(params.Δθf))# Right Ascension Separation


0.0004469491942075327


(l_12 = 0.03, l_13 = 6.0, l_14 = 6.03, l_15 = 4.0, l_16 = 5.0, Δx0 = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Δaf = 0, u_min = 0.2, u_max = 1.0, Δa_min = -0.002165084206620048, Δa_max = 0.002165084206620048, Δθf = [0.18849555921538758, 37.69911184307752, 37.88760740229291, 25.132741228718345, 31.41592653589793], ΔΩf = [8.424793830301574e-5, 0.016849587660603148, 0.016933835598906165, 0.011233058440402099, 0.014041323050502624])

In [13]:
#Discretized Optimal Control Problem Parameters
const Nx = size(Ad,1)     # number of state
const Nu = size(Bd,2)     # number of controls
const Tfinal =5000#Int(round(800*P_mks)) # final time (number of orbits)
const Nt = Int(Tfinal/h)+1    # number of time steps
const Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));

#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]
U1 = [sparse(kron(Diagonal(I,Nt), sparse([I spzeros(1,(Nu-1+Nx))]))) for i = 1:num_sat] #Matrix that picks out all u of sat 1
U2 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,i) I spzeros(1,(Nu-i)+Nx-1)]))) for i = 1:num_sat] #Matrix that picks out all u of sat n
X1 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,Nu) spzeros(1,(3*(i-1))) I spzeros(1,3*(num_sat)-1-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x1 = Δθ
X2 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,Nu) spzeros(1,1) spzeros(1,(3*(i-1))) I spzeros(1,3*(num_sat)-2-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x2 = Δa
X3 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,Nu) spzeros(1,2) spzeros(1,(3*(i-1))) I spzeros(1,3*(num_sat)-3-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x3 = ΔΘ

X1F = spzeros(num_sat,(Nu+Nx)*Nt);  #Matrix that picks out x1f = Δθf
X2F = spzeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x2f = Δaf
# X3F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x3f = ΔΘf

for i = num_sat:-1:1
    X1F[num_sat-i+1,end-(i)*3+1] = 1.0
    X2F[num_sat-i+1,end-(i)*3+2] = 1.0
#     X3F[num_sat-i+1,end-(i)*3+3] = 1.0
end


In [14]:

# z = Variable(Nz)
b_c = zeros(Nt + Nt*num_sat + Nt*num_sat)
m_c = length(b_c)
z = Variable(Nz+m_c)
# t_c = Variable(m_c)

# linear cost term
c_c = [zeros(Nz);ones(m_c)]
function LP_controller(a,Δx, params)
    #Dynamics equality constraints C*z = d
    Ad, Bd = c2d(a, h, num_sat)
    C = sparse([[Bd -I spzeros(Nx,(Nt-1)*(Nu+Nx))]; spzeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [Ad Bd]) spzeros((Nt-1)*Nx,Nx)] + [spzeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[spzeros(Nx,Nu) Diagonal(-I,Nx)])]])
    d = sparse([-Ad*Δx; spzeros(Nx*(Nt-1))]);

    
    #Convex Formulation
    Ac = U1[1] 
    for i in 1:num_sat
        Ac  = [Ac; U2[i];]
    end

    for i in 1:num_sat
        Ac  = [Ac; X1[i]]
        b_c[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
    end
    
    # inequalities constraints
    G_c = [ Ac -I(m_c);
       -Ac -I(m_c)]

    h_c = [b_c; -b_c]
    problem = minimize(c_c'*z)
    problem.constraints += G_c*z <= h_c
    
#     problem = minimize(ones(m_c)'*t_c)
#     problem.constraints += t_c >= (Ac*z-b)
#     problem.constraints += t_c >= -(Ac*z-b)
    problem.constraints += [C*z[1:Nz] == d] #dynamics constraints

    for i = 1:num_sat
        problem.constraints += [X2F[i,:]'*z[1:Nz] == params.Δaf] #final altitude constraint
    #     problem.constraints += [X3F[i,:]'*z == ΔΩf] #final altitude constraint
        problem.constraints += [U1[i]*z[1:Nz] >= params.u_min; U1[i]*z[1:Nz] <= params.u_max] #drag constraints
        problem.constraints += [U2[i]*z[1:Nz] >= params.u_min; U2[i]*z[1:Nz] <= params.u_max] #drag constraints
        problem.constraints += [X2[i]*z[1:Nz] >= params.Δa_min; X2[i]*z[1:Nz] <= params.Δa_max]; #altitude constraints
    end
    
    solve!(problem, Mosek.Optimizer(), warmstart=true; silent_solver = true)#Mosek.Optimizer)
    # Optimizer
#     optimizer = GLPK.Optimizer()
#     MOI.set(optimizer, MOI.RawParameter("msg_lev"), GLPK.GLP_MSG_ON)
    #     solve!(problem,optimizer; silent_solver = false)
    
    u1 = zeros(size(U1[1])[1],num_sat) # sat 1 control
    u2 = zeros(size(U2[1])[1],num_sat) # sat n control
    for i = 1:num_sat
        u1[:,i] = U1[i]*z.value[1:Nz];
        u2[:,i] = U2[i]*z.value[1:Nz];
    end
    return [u1[1:h,:],u2[1:h,:]]
end

LP_controller (generic function with 1 method)

In [15]:
r_s0 = [a_mks*(1-ecc)*10^3; 0; 0]
v_s0 = [0; cosd(inc)*sqrt((μ_mks*(1+ecc))/(a_mks*(1-ecc)))*10^3; sind(inc)*sqrt((μ_mks*(1+ecc))/(a_mks*(1-ecc)))*10^3]

rv_to_kepler(r_s0, v_s0)


KeplerianElements{Int64, Float64}:
           Epoch :    0 (-4713-11-24T12:00:00)
 Semi-major axis : 6928.14  km
    Eccentricity :    0.005
     Inclination :   98.0   °
            RAAN :    0.0   °
 Arg. of Perigee :    0.0   °
    True Anomaly :    0.0   °

In [16]:
# initial conditions
r_s0 = [a_mks*(1-ecc); 0; 0]
v_s0 = [0; cosd(inc)*sqrt((μ_mks*(1+ecc))/(a_mks*(1-ecc))); sind(inc)*sqrt((μ_mks*(1+ecc))/(a_mks*(1-ecc)))]
r_s1 = [a_mks*(1-ecc); 0; 0]
v_s1 = [0; cosd(inc)*sqrt((μ_mks*(1+ecc))/(a_mks*(1-ecc))); sind(inc)*sqrt((μ_mks*(1+ecc))/(a_mks*(1-ecc)))]
x0 = repeat([r_s0; v_s0; r_s1; v_s1],num_sat);

# initialization results
u1hist = zeros(Tfinal,num_sat)
u2hist = zeros(Tfinal,num_sat)
Δθhist = zeros(Tfinal,num_sat)
Δθunhist = zeros(Tfinal,num_sat)
Δahist = zeros(Tfinal,num_sat)
ΔΩhist = zeros(Tfinal,num_sat)
a1hist = zeros(Tfinal,num_sat)
a2hist = zeros(Tfinal,num_sat)
θ1hist = zeros(Tfinal,num_sat)
θ2hist = zeros(Tfinal,num_sat)
Ω1hist = zeros(Tfinal,num_sat)
Ω2hist = zeros(Tfinal,num_sat)
tp_hist = zeros(1,Tfinal)

h_traj = 30 # integrate traj 1/5 Hz
T = P_mks#*h # integrate traj for n orbital period
x_hist = zeros(length(x0),Int(round(T/h_traj))*Tfinal+1)
t_hist = zeros(1,Int(round(T/h_traj))*Tfinal+1)

# find first controller
xc = zeros(size(Ad,1))
for k = 0:num_sat-1
    xc[1+k*3:3+k*3,1] = cartesian_to_ΔθaΩ(mks_to_orbital(x0[1+k*12:6+k*12]),mks_to_orbital(x0[7+k*12:12+k*12]))
end


uk = LP_controller(a_vec, xc, params)

# store first step
x_hist[:,1] = x0
for k = 0:num_sat-1
    Δθhist[1,k+1] = xc[1+k*3]
    Δahist[1,k+1] = xc[2+k*3]
    ΔΩhist[1,k+1] = xc[3+k*3]
    u1hist[1,k+1] = uk[1][k+1]
    u2hist[1,k+1] = uk[2][k+1]
    a1hist[1,k+1] = cartesian_to_θaΩ(mks_to_orbital(x0[1+12*k:6+12*k]))[2]
    a2hist[1,k+1] = cartesian_to_θaΩ(mks_to_orbital(x0[7+12*k:12+12*k]))[2]
    θ1hist[1,k+1] = cartesian_to_θaΩ(mks_to_orbital(x0[1+12*k:6+12*k]))[1]
    θ2hist[1,k+1] = cartesian_to_θaΩ(mks_to_orbital(x0[7+12*k:12+12*k]))[3]
    Ω1hist[1,k+1] = cartesian_to_θaΩ(mks_to_orbital(x0[1+12*k:6+12*k]))[1]
    Ω2hist[1,k+1] = cartesian_to_θaΩ(mks_to_orbital(x0[7+12*k:12+12*k]))[3]
end

# initialize index
kk = 0 # traj steps count
mm = zeros(num_sat,1) # Wrapping matrix
ll = 0 # orbit counter
for k = 1:Tfinal-1 # control horizon
    ss = 0 # orbit counter for indexing
    for orbit = 1:1 # save results for each orbit
        ll += 1 # increase orbit counter
        for time = 0:h_traj:P_mks # loop for simulations for entire orbit
            kk += 1 # increase traj steps counter
            
            for jj = 0:num_sat-1 # loop for simulation for each couple of satellites
                u = [uk[1][jj*h+orbit],uk[2][jj*h+orbit]] # pick control stored
                x_hist[1+jj*12:12+jj*12,kk+1] .= sat_dynamics_rk4(x_hist[1+jj*12:12+jj*12,kk],u,h_traj) #simulate couple of sats
            end
            t_hist[kk+1] = time # save simulation time
        end
        
        for jj = 0:num_sat-1 # transform from cartesian to ΔθaΩ coordinates for each couple of sats
            xc[1+jj*3:3+jj*3,1] = cartesian_to_ΔθaΩ(mks_to_orbital(x_hist[1+jj*12:6+jj*12,kk+1]),mks_to_orbital(x_hist[7+jj*12:12+jj*12,kk+1]))
        end

        #save results
        for jj = 0:num_sat-1
            Δθhist[ll+1,jj+1] = xc[1+jj*3]
            Δahist[ll+1,jj+1] = xc[2+jj*3]
            if inc < 90
                if xc[3+jj*3] > 0  #this if avoids the inf difference switch
                    ΔΩhist[ll+1,jj+1] = 0 
                else
                    ΔΩhist[ll+1,jj+1] = xc[3+jj*3]
                end
            else
                if xc[3+jj*3] < 0  #this if avoids the inf difference switch
                    ΔΩhist[ll+1,jj+1] = 0 
                else
                    ΔΩhist[ll+1,jj+1] = xc[3+jj*3]
                end
            end

            u1hist[ll+1,jj+1] = uk[1][jj*h+orbit]
            u2hist[ll+1,jj+1] = uk[2][jj*h+orbit]

            # angle wrapping, adjust angle for linearized dynamics of controller
            if abs(Δθhist[ll+1,jj+1]-Δθhist[ll,jj+1]) > pi-0.1
                mm[jj+1] =mm[jj+1]+ 2
                println("mm change ", abs(Δθhist[ll+1,jj+1]-Δθhist[ll,jj+1]), " - ",mm[jj+1] )
            end

            xc[1+jj*3] = mm[jj+1]*pi + xc[1+jj*3]
            Δθunhist[ll+1,jj+1] = xc[1+jj*3]
        end

        tp_hist[ll+1] = ll # store orbit time
        ss += 1*(num_sat-1) # increase index counter
    end
    
    # evaluate mean apoapsis between each couple of sats to update linearized dynamics
    a1 = zeros(num_sat,1); a2 = zeros(num_sat,1); a_upd = zeros(num_sat,1)
    for jj = 0:num_sat-1
        a1[jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[1+12*jj:6+12*jj,kk+1]))[2]
        a2[jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[7+12*jj:12+12*jj,kk+1]))[2]
        a_upd[jj+1] = mean([a1[jj+1],a2[jj+1]]) 
        a1hist[ll+1,jj+1] = a1[jj+1]
        a2hist[ll+1,jj+1] = a2[jj+1]
        θ1hist[ll+1,jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[1+12*jj:6+12*jj,kk+1]))[1]
        θ2hist[ll+1,jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[7+12*jj:12+12*jj,kk+1]))[1]
        Ω1hist[ll+1,jj+1] =cartesian_to_θaΩ(mks_to_orbital(x_hist[1+12*jj:6+12*jj,kk+1]))[3]
        Ω2hist[ll+1,jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[7+12*jj:12+12*jj,kk+1]))[3]
    end
    
    # evaluate controller
    uk = LP_controller(a_upd, xc, params)
#     println("k ",k, " a ", a_upd)#" u ", uk, "
    if k % 1 == 0
        println("k ",k, " - ",maximum(abs.(Δθunhist[ll+1,:].-params.Δθf[1:num_sat])), " - ", maximum(abs.(Δahist[ll+1,:].-params.Δaf[1])).*a_mks)
    end
    if maximum(abs.(Δθunhist[ll+1,:].-params.Δθf[1:num_sat]))<1e-3 && maximum(abs.(Δahist[ll+1,:].-params.Δaf[1]))<1e-4 # abs(Δθunhist[ll+1,:].-params.Δθf[1:num_sat])
        println("Reached Final Conditions")
        # Resize Results
        Δθhist = Δθhist[1:ll+1,:]
        Δahist = Δahist[1:ll+1,:]
        ΔΩhist = ΔΩhist[1:ll+1,:]
        u1hist = u1hist[1:ll+1,:]
        u2hist = u2hist[1:ll+1,:]
        a1hist = a1hist[1:ll+1,:]
        a2hist = a2hist[1:ll+1,:]
        θ1hist = θ1hist[1:ll+1,:]
        θ2hist = θ2hist[1:ll+1,:]
        Ω1hist = Ω1hist[1:ll+1,:]
        Ω2hist = Ω2hist[1:ll+1,:]
        tp_hist = tp_hist[1:ll+1]
        Δθunhist = Δθunhist[1:ll+1,:]

        break 
    end
end

x_hist = x_hist[:,1:kk]
t_hist = t_hist[:,1:kk]
print(kk)

k 1 - 37.887602986812105 - 0.006442624132153417
k 2 - 37.88758974241131 - 0.012884981406557309
k 3 - 37.8875676703926 - 0.019327060940116093
k 4 - 37.88753677251455 - 0.025768793686822423
k 5 - 37.887497050985466 - 0.032210045878978744
k 6 - 37.88744850845495 - 0.038650618994891854
k 7 - 37.887391148004035 - 0.04509025713374886
k 8 - 37.88732497313391 - 0.051528661801520555
k 9 - 37.88724998775322 - 0.057965514047127055
k 10 - 37.88716619616428 - 0.06440050285817171
k 11 - 37.887073603047824 - 0.07083335812405243
k 12 - 37.886972213446825 - 0.07726388630580928
k 13 - 37.886862032749235 - 0.08369200650132952
k 14 - 37.886743066669574 - 0.09011778418071324
k 15 - 37.886615321229804 - 0.09654145995967367
k 16 - 37.88647880273904 - 0.1029634707734614
k 17 - 37.88633351777274 - 0.1093844615675966
k 18 - 37.88617947315077 - 0.11580528548785594
k 19 - 37.8860166759148 - 0.1222269914742051
k 20 - 37.885845133304954 - 0.12865079876794153
k 21 - 37.88566485273561 - 0.1350780589089308
k 22 - 37.8

k 176 - 37.75090177940982 - 1.1379660930725701
k 177 - 37.74933166662542 - 1.1444438847562537
k 178 - 37.7477534423696 - 1.1507802077258105
k 179 - 37.74616715258768 - 1.1569913612909755
k 180 - 37.74457284326814 - 1.1630971835958837
k 181 - 37.742970560369244 - 1.1691198733203327
k 182 - 37.74136034974041 - 1.175082711757917
k 183 - 37.73974225703846 - 1.181008759017486
k 184 - 37.73811632763879 - 1.186919599612733
k 185 - 37.736482606542225 - 1.1928342109571544
k 186 - 37.73484113827777 - 1.1987680215482344
k 187 - 37.73319196680246 - 1.2047322146331623
k 188 - 37.73153513539872 - 1.2107333192354985
k 189 - 37.72987068657061 - 1.2167731131803243
k 190 - 37.728198661939615 - 1.2228488438067127
k 191 - 37.72651910214128 - 1.2289537531399704
k 192 - 37.72483204672342 - 1.2350778747082218
k 193 - 37.72313753404659 - 1.2412090521044707
k 194 - 37.72143560118753 - 1.247334114651672
k 195 - 37.719726283845866 - 1.2534401342429742
k 196 - 37.718009616254584 - 1.259515681286639
k 197 - 37.716

k 351 - 37.34314587048936 - 2.255325627102511
k 352 - 37.33997543658529 - 2.2601841581790856
k 353 - 37.336799340094004 - 2.2649953565500454
k 354 - 37.333617694551464 - 2.2697692032724124
k 355 - 37.330430610018986 - 2.2745226345850726
k 356 - 37.32723819295205 - 2.27928016588497
k 357 - 37.32404054607548 - 2.28407398606214
k 358 - 37.32083776826405 - 2.2889434885243616
k 359 - 37.31762995442691 - 2.2939342420112125
k 360 - 37.314417195394746 - 2.2990964426846716
k 361 - 37.3111995778082 - 2.304482923477944
k 362 - 37.30797718400676 - 2.3101468274989974
k 363 - 37.30475009191688 - 2.31613907563752
k 364 - 37.30151837493933 - 2.3225057740337194
k 365 - 37.298282101835326 - 2.3292857130046603
k 366 - 37.295041336611824 - 2.3365081056663772
k 367 - 37.291796138406575 - 2.3441907015024372
k 368 - 37.28854656137363 - 2.352338388687257
k 369 - 37.285292654570945 - 2.3609423707898065
k 370 - 37.282034461850785 - 2.3699799697087363
k 371 - 37.278772021754904 - 2.379415070208644
k 372 - 37.275

k 527 - 36.65155664089459 - 3.4753167262248423
k 528 - 36.646670595892665 - 3.4806372400530288
k 529 - 36.64178563543281 - 3.4856484323441275
k 530 - 36.63690207103412 - 3.4904319172917773
k 531 - 36.63202019959664 - 3.4950661673647536
k 532 - 36.62714030280767 - 3.4996228231653115
k 533 - 36.62226264656612 - 3.504163584374376
k 534 - 36.617387480428555 - 3.5087378686791526
k 535 - 36.61251503708025 - 3.5133813717262887
k 536 - 36.60764553183549 - 3.5181155980019203
k 537 - 36.6027791621712 - 3.522948365289956
k 538 - 36.59791610729766 - 3.527875219222985
k 539 - 36.593056527770166 - 3.5328816324159904
k 540 - 36.58820056514464 - 3.5379458115989073
k 541 - 36.58334834167943 - 3.5430418969052075
k 542 - 36.57849996008527 - 3.5481433151475583
k 543 - 36.57365550332359 - 3.553226042788171
k 544 - 36.568815034453536 - 3.558271545498375
k 545 - 36.56397859652622 - 3.563269189919211
k 546 - 36.55914621252472 - 3.5682179636897295
k 547 - 36.5543178853476 - 3.5731273936236665
k 548 - 36.549493

k 704 - 35.6586157326165 - 4.6405509061620345
k 705 - 35.65192842754024 - 4.64464702313909
k 706 - 35.64525288492442 - 4.6491710147539305
k 707 - 35.63858957223091 - 4.6541686994204525
k 708 - 35.63193892589441 - 4.659682717648647
k 709 - 35.62530135040167 - 4.66574999459652
k 710 - 35.61867721741973 - 4.6723991543199626
k 711 - 35.612066864974494 - 4.679648125744781
k 712 - 35.605470596681926 - 4.687502169255107
k 713 - 35.59888868103495 - 4.695952517988641
k 714 - 35.592321350749465 - 4.704975779317927
k 715 - 35.585768802173114 - 4.714534178202034
k 716 - 35.579231194761114 - 4.724576654757014
k 717 - 35.57270865062283 - 4.735040758295274
k 718 - 35.5662012541424 - 4.745855214653904
k 719 - 35.55970905167648 - 4.756942989102664
k 720 - 35.55323205133098 - 4.768224628481952
k 721 - 35.54677022281813 - 4.779621645939369
k 722 - 35.54032349739397 - 4.791059711081244
k 723 - 35.533891767875375 - 4.802471427891817
k 724 - 35.527474888735085 - 4.813798520227899
k 725 - 35.521072676272155 

...

k 3952 - 0.6403970363503575 - 0.9463389969328132
k 3953 - 0.638734633795238 - 0.9449845364286773
k 3954 - 0.6370801174889138 - 0.9421412251196419
k 3955 - 0.6354350808784943 - 0.937843311529081
k 3956 - 0.633801056254157 - 0.9322639565400296
k 3957 - 0.6321795085226611 - 0.9257112730752377
k 3958 - 0.6305718295402443 - 0.9186034032490953
k 3959 - 0.6289793329429543 - 0.911425444718284
k 3960 - 0.6274029211201793 - 0.9053141362063573
k 3961 - 0.6258427560132915 - 0.9009100104832133
k 3962 - 0.6242990183230432 - 0.8980453010891658
k 3963 - 0.622772302828146 - 0.8968325769906949
k 3964 - 0.621263337075959 - 0.8970918178609162
k 3965 - 0.6197730114018682 - 0.8986718842211259
k 3966 - 0.6183023315254772 - 0.9011258677900749
k 3967 - 0.6168524764615526 - 0.904102672076375
k 3968 - 0.6154246394393041 - 0.9072500388418367
k 3969 - 0.6140197986840974 - 0.9105896553646046
k 3970 - 0.6126384303117263 - 0.9141683165519995
k 3971 - 0.611280687552096 - 0.9178133567176034
k 3972 - 0.6099466272038043 

k 4120 - 0.42713178356704873 - 1.0016118753915924
k 4121 - 0.42599873142507505 - 1.0189800014123838
k 4122 - 0.4248418957675497 - 1.0344491968355456
k 4123 - 0.4236608018526411 - 1.0471556301567595
k 4124 - 0.42245504869586625 - 1.056372034600201
k 4125 - 0.42122431402360405 - 1.0615647925731442
k 4126 - 0.41996835923237086 - 1.0624518556098734
k 4127 - 0.4186870341072293 - 1.0590518420301887
k 4128 - 0.41738028103892333 - 1.0517123883971755
k 4129 - 0.41604813849122024 - 1.0411064663304521
k 4130 - 0.4146907435106897 - 1.02818917920807
k 4131 - 0.4133083331410603 - 1.0141139869603748
k 4132 - 0.411901244698484 - 1.000115126893679
k 4133 - 0.4104699149684521 - 0.9873705024409999
k 4134 - 0.40901487848683615 - 0.9768647435717273
k 4135 - 0.40753676514754034 - 0.9692740944951641
k 4136 - 0.4060362974238956 - 0.964892551323805
k 4137 - 0.40451428748929175 - 0.9636124507436119
k 4138 - 0.402971634473424 - 0.9649635792429301
k 4139 - 0.4014093219999211 - 0.9682045947920439
k 4140 - 0.399828

k 4287 - 0.20394912968137646 - 0.9796375507462831
k 4288 - 0.20287040715559357 - 0.9811649808497233
k 4289 - 0.20180726126964998 - 0.982578112129832
k 4290 - 0.20075871613106955 - 0.9837857360046643
k 4291 - 0.1997237341659357 - 0.984784296155642
k 4292 - 0.19870122131137435 - 0.9856401462834866
k 4293 - 0.19769003225653847 - 0.9864560986660275
k 4294 - 0.1966889757712451 - 0.9873308977201025
k 4295 - 0.19569682022682144 - 0.9883217727703276
k 4296 - 0.19471229945014557 - 0.9894194494809394
k 4297 - 0.19373411904724946 - 0.9905420847686999
k 4298 - 0.19276096328481174 - 0.9915501547468922
k 4299 - 0.19179150253365407 - 0.9922793385374301
k 4300 - 0.19082440117340127 - 0.9925840077292214
k 4301 - 0.1898583257527804 - 0.992381061046571
k 4302 - 0.1888919531198141 - 0.9916832259470917
k 4303 - 0.1879239781979436 - 0.9906127923545327
k 4304 - 0.18695312110111217 - 0.9893907193015983
k 4305 - 0.18597813335327373 - 0.9883013568468098
k 4306 - 0.18499780309431202 - 0.9876385265062416
k 4307 -

k 4452 - 0.02463050994923957 - 0.48311987284068564
k 4453 - 0.024617527516110682 - 0.4746786198998445
k 4454 - 0.02460510885447728 - 0.467311344324852
k 4455 - 0.024592843746162052 - 0.4604218248144056
k 4456 - 0.024580322749828132 - 0.4534325892725918
k 4457 - 0.02456713797041743 - 0.4458748287255047
k 4458 - 0.02455288338463954 - 0.43745293398420615
k 4459 - 0.024537154894143126 - 0.42807351524564446
k 4460 - 0.024519550326161932 - 0.41783596435277603
k 4461 - 0.024499669607617958 - 0.40698914232399475
k 4462 - 0.024477115302437125 - 0.3958650763012252
k 4463 - 0.02445149363146984 - 0.3848043784696937
k 4464 - 0.02442241600538586 - 0.37408876553953496
k 4465 - 0.024389501010097042 - 0.36389354461345863
k 4466 - 0.024352376709810913 - 0.35426790284918447
k 4467 - 0.024310683089820206 - 0.34514444931598687
k 4468 - 0.024264074453162532 - 0.3363731198671538
k 4469 - 0.02421222161509129 - 0.3277696067622592
k 4470 - 0.024154813795463276 - 0.31916592062545196
k 4471 - 0.024091560180429017

k 4612 - 0.005358036525992066 - 0.029411979339557
k 4613 - 0.005231003219965141 - 0.027647538006634914
k 4614 - 0.005109732911501963 - 0.02898539778259271
k 4615 - 0.004994580148223682 - 0.030248303475584985
k 4616 - 0.004885862803236385 - 0.031071646526876658
k 4617 - 0.004783860368238996 - 0.031139873441485666
k 4618 - 0.004688813157270738 - 0.030267336226034287
k 4619 - 0.004600922360586346 - 0.028459237117125603
k 4620 - 0.004520350765501696 - 0.02593737628253038
k 4621 - 0.004447223871341666 - 0.02312246687932219
k 4622 - 0.004381631090259519 - 0.02057395180375352
k 4623 - 0.004323626752497756 - 0.021365266940830564
k 4624 - 0.004273230717070919 - 0.0214899893791249
k 4625 - 0.004230552202229149 - 0.020005432165033254
k 4626 - 0.004195529489295441 - 0.023482256170426666
k 4627 - 0.004167967324094052 - 0.028765086203697218
k 4628 - 0.004147745885370757 - 0.035455710631298196
k 4629 - 0.004134709564764777 - 0.042945156154468235
k 4630 - 0.004128667335824759 - 0.050499880607867975
k 

In [ ]:
# pgfplotsx()
save_res =1

mylabel = String[]
mylabel1 = String[]
mylabel2 = String[]
for i = 1:num_sat
    append!(mylabel,["Sat. 1-"*string(i+1)])
    append!(mylabel1,["Sat. 1"])
    append!(mylabel2,["Sat. "*string(i+1)])
end

pgfplotsx()

# plotly()

In [ ]:
pl1 = plot(tp_hist[1:end],u1hist[1:end,1],label=permutedims(mylabel1), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11, tex_output_standalone = true)#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],u2hist[1:end,:],label=permutedims(mylabel2), linewidth=2) #,ls=:dot
plot!(xlabel="Orbit Number", ylabel = "Drag Ratio")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)
if save_res ==1
     savefig(pl1,  "Plots/ControlBox.tikz")    # produces a tikzpicture environment that can be included in other documents
     savefig(pl1, "Plots/ControlBox.pdf")
     savefig(pl1,  "Plots/ControlBox.tex") 
end

In [ ]:
plot(t_hist[1:20:end],x_hist[1,1:20:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:20:end],x_hist[2,1:20:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:20:end],x_hist[3,1:20:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:20:end],x_hist[7,1:20:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:20:end],x_hist[8,1:20:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:20:end],x_hist[9,1:20:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

In [ ]:
# plot(tp_hist, a1hist[:,1]*a_mks.-Re_mks,label=permutedims(mylabel1), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11)
# plot!(tp_hist, a2hist*a_mks.-Re_mks,label=permutedims(mylabel2),label=permutedims(mylabel2), linewidth=2)

println("Altitude drop ", (a1hist[1,1]*a_mks.-Re_mks)-(a1hist[end,1]*a_mks.-Re_mks), " km")

println("Tot Time ", (t_hist[end]*tp_hist[end])/(60*60*24*30), " months")
pl2 = plot(tp_hist[1:end],a1hist[:,1]*a_mks.-Re_mks,label=permutedims(mylabel1), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11, tex_output_standalone = true)#, ylim=[0, 500])#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],a2hist*a_mks.-Re_mks,label=permutedims(mylabel2), linewidth=2)
plot!(xlabel="Orbit Number", ylabel = "Altitude")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)
if save_res ==1
    savefig(pl2,  "Plots/AltitudeBox.tikz")    # produces a tikzpicture environment that can be included in other documents
    savefig(pl2,  "Plots/AltitudeBox.tex")
    savefig(pl2, "Plots/AltitudeBox.pdf")
end

In [ ]:
pl=plot(tp_hist[1:end],broadcast.(rad2deg,Δθunhist[1:end,:]),label=permutedims(mylabel), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:topleft,legendfontsize=11, tex_output_standalone = true)
plot!(xlabel="Orbit Number", ylabel = "Argument of Latitude Difference, deg")
if save_res ==1
    savefig(pl,  "Plots/ArgumentofLatDiffUnwrappedBox.tikz")    # produces a tikzpicture environment that can be included in other documents
    savefig(pl,  "Plots/ArgumentofLatDiffUnwrappedBox.tex")
    savefig(pl, "Plots/ArgumentofLatDiffUnwrappedBox.pdf")
end

In [ ]:
pl3=plot(tp_hist[1:end],broadcast.(rad2deg,Δθhist[1:end,:]),label=permutedims(mylabel), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:topleft,legendfontsize=11, tex_output_standalone = true)
plot!(xlabel="Orbit Number", ylabel = "Argument of Latitude Difference, deg")
if save_res ==1
    savefig(pl3,  "Plots/ArgumentofLatDiffBox.tikz")    # produces a tikzpicture environment that can be included in other documents
    savefig(pl3,  "Plots/ArgumentofLatDiffBox.tex")
    savefig(pl3, "Plots/ArgumentofLatDiffBox.pdf")
end

In [ ]:
println("Tot Time ", t_hist[end]/(60*60*24), " days")
pl = plot(tp_hist[1:end],broadcast.(rad2deg,θ1hist[1:end,1]),label=permutedims(mylabel1), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11, tex_output_standalone = true)#, ylim=[0, 500])#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],broadcast.(rad2deg,θ2hist[1:end,:]),label=permutedims(mylabel2), linewidth=2)
plot!(xlabel="Orbit Number", ylabel = "Argument of Latitude, deg")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)
if save_res ==1
    savefig(pl,  "Plots/ArgumentofLatBox.tikz")    # produces a tikzpicture environment that can be included in other documents
    savefig(pl,  "Plots/ArgumentofLatBox.tex")
    savefig(pl, "Plots/ArgumentofLatBox.pdf")
end

In [ ]:
pl = plot(tp_hist[1:end],a_mks*Δahist[1:end,:], label=permutedims(mylabel), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:bottomleft,legendfontsize=11, tex_output_standalone = true)
plot!(xlabel="Orbit Number", ylabel = "Semi-major axis Difference, km")
if save_res ==1
    savefig(pl,  "Plots/SemimajorAxisBox.tikz")
    savefig(pl,  "Plots/SemimajorAxisBox.tex")
    savefig(pl, "Plots/SemimajorAxisBox.pdf")
end

In [ ]:
pl4 = plot(tp_hist[1:end],broadcast.(rad2deg,ΔΩhist[1:end,:]),label=permutedims(mylabel), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11, tex_output_standalone = true)
plot!(xlabel="Orbit Number", ylabel = "RAAN Difference, deg",ylim = [-5,1])
if save_res ==1
    savefig(pl4,  "Plots/RAANDiffBox.tikz")
    savefig(pl4,  "Plots/RAANDiffBox.tex")
    savefig(pl4, "Plots/RAANDiffBox.pdf")
end

In [ ]:

pl = plot(tp_hist[1:end],broadcast.(rad2deg,Ω1hist[1:end,1]),label=permutedims(mylabel1), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11, tex_output_standalone = true)#, ylim=[0, 500])#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],broadcast.(rad2deg,Ω2hist[1:end,:]),label=permutedims(mylabel2), linewidth=2)
plot!(xlabel="Orbit Number", ylabel = "RAAN, deg")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)
if save_res ==1
    savefig(pl,  "Plots/RAANBox.tikz")    # produces a tikzpicture environment that can be included in other documents
    savefig(pl,  "Plots/RAANBox.tex")
    savefig(pl, "Plots/RAANBox.pdf")
end

In [ ]:
pl = plot(tp_hist[1:end],a_mks*ΔΩhist[1:end,:],label=permutedims(mylabel), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11, tex_output_standalone = true)
plot!(xlabel="Orbit Number", ylabel = "Spherical Distance, km")
if save_res ==1
    savefig(pl,  "Plots/SphericalDistanceBox.tikz")
    savefig(pl,  "Plots/SphericalDistanceBox.tex")
    savefig(pl, "Plots/SphericalDistancBox.pdf")
end

In [ ]:
using LaTeXStrings
pl1 = plot(tp_hist[1:end],u1hist[1:end,1],label=permutedims(mylabel1), linewidth=1.5, xtickfontsize=6, ytickfontsize=6,yguidefontsize=9,xguidefontsize=9,legend=:outertop,legendfontsize=7, tex_output_standalone = true, legend_columns=-1)#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],u2hist[1:end,:],label=permutedims(mylabel2), linewidth=1.5) #,ls=:dot
plot!(xlabel=nothing, ylabel = "Drag Ratio")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)"Orbit Number"

pl2 = plot(tp_hist[1:end],a1hist[:,1]*a_mks.-Re_mks,label=nothing, linewidth=1.5, xtickfontsize=6, ytickfontsize=6,yguidefontsize=9,xguidefontsize=9,legend=:top,legendfontsize=7, tex_output_standalone = true)#, ylim=[0, 500])#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],a2hist*a_mks.-Re_mks,label=nothing, linewidth=1.5)
plot!(xlabel=nothing, ylabel = L"a-$R_E$ (km)")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)

pl3 = plot(tp_hist[1:end],broadcast.(rad2deg,ΔΩhist[1:end,:]),label=permutedims(mylabel), linewidth=1.5, xtickfontsize=6, ytickfontsize=6,yguidefontsize=9,xguidefontsize=9,legend=:outertop,legendfontsize=7, tex_output_standalone = true,legend_columns=-1)
plot!(xlabel=nothing, ylabel = L"$\Delta \Omega$ (deg)")

pl4=plot(tp_hist[1:end],broadcast.(rad2deg,Δθhist[1:end,:]),label=nothing, linewidth=1.5, xtickfontsize=6, ytickfontsize=6,yguidefontsize=9,xguidefontsize=9,legend=:outertop,legendfontsize=7, tex_output_standalone = true,legend_columns=-1)
plot!(xlabel="Orbit Number", ylabel = L"$\Delta \theta$ (deg)")#"\$\\Delta \\bar{\\theta}, deg\$")




pl5 = plot(pl1,pl2,pl3,pl4, layout = (4,1), margins=(10,:px), size=(300,500))
if save_res ==1
    savefig(pl5,  "Plots/ResultsBox.tikz")
    savefig(pl5,  "Plots/ResultsBox.tex")
    savefig(pl5, "Plots/ResultsBox.pdf")
end

In [ ]:
# Final RAAN
broadcast.(rad2deg,ΔΩhist[:,end]')

println("Targeted Δθf ",  broadcast.(rad2deg, params.Δθf[1:num_sat]), " deg" )
println("Reached Δθf ",  broadcast.(rad2deg, Δθunhist[end,:]), " deg" )

println("Reached Δθf ",  broadcast.(rad2deg, Δθhist[end,:]), " deg" )


println("Targeted ΔΩ ",  broadcast.(rad2deg, params.ΔΩf[1:num_sat]), " deg" )
println("Reached ΔΩ ",  broadcast.(rad2deg, ΔΩhist[end,:]), " deg" )

println("θ1f",  broadcast.(rad2deg, θ1hist[end,:]), " deg" )
println("θ2f",  broadcast.(rad2deg, θ2hist[end,:]), " deg" )

println("Ω1f",  broadcast.(rad2deg, Ω1hist[end,:]), " deg" )
println("Ω2f",  broadcast.(rad2deg, Ω2hist[end,:]), " deg" )

println("Final Error ΔΩ ", broadcast.(rad2deg, (params.ΔΩf[1:num_sat] .- ΔΩhist[end,:])), " deg" )
println("Spherical distance ", a_mks*ΔΩhist[end,:], " km")
println("Spherical distance ", a_mks*Δθhist[end,:], " km")
# println("Final Azimut Sat 1 ", broadcast.(rad2deg, ϕ_soln1[:,end]), " deg")
# println("Final Azimut Sat Coupled ", broadcast.(rad2deg, ϕ_soln2[:,end]), " deg")

# println("Final Elevation Sat 1 ", broadcast.(rad2deg, θ_soln1[:,end]), " deg")
# println("Final Elevation Sat Coupled ", broadcast.(rad2deg, θ_soln2[:,end]), " deg")
sqrt(114.95^2+1312.09^2)

In [ ]:
angle = asin(norm(cross(x_hist[1:3,end],x_hist[7:9,end]))/(norm(x_hist[1:3,end])*norm(x_hist[7:9,end])))
println(rad2deg(angle))
# for i = 1:num_sat
#     println("Couple ",i ," Radius first spacecraft ", norm(x_hist[1:3,end])," - Radius second spacecraft ", norm(x_hist[7+(12*(i-1)):9+(12*(i-1)),end]))
#     angle = acos((dot(x_hist[1:3,end],x_hist[7+(12*(i-1)):9+(12*(i-1)),end]))/(norm(x_hist[1:3,end])*norm(x_hist[7+(12*(i-1)):9+(12*(i-1)),end])))
#     println("Couple ",i ," Angle ", rad2deg(angle), " deg")
#     distance = norm(x_hist[1:3,end])*angle
#     println("Couple ",i ," Sperical Distance ", distance, " km")
# end
println(x_hist[7:12,end])


# rv_to_kepler(x_hist[1:3].*10^3, x_hist[4:6].*10^3)
oe0 = rv_to_kepler(x_hist[1:3,end].*10^3, x_hist[4:6,end].*10^3)
r0, v0 = kepler_to_rv(oe0.a, oe0.e, oe0.i, oe0.Ω, oe0.f+oe0.ω, 0)#oe0.f+oe0.ω)

oe1 = rv_to_kepler(x_hist[7:9,end].*10^3, x_hist[10:12,end].*10^3)
r1, v1 = kepler_to_rv(oe1.a, oe1.e, oe1.i, oe1.Ω, oe1.f+oe1.ω, 0)#oe1.f+oe1.ω)

oe2 = rv_to_kepler(x_hist[19:21,end].*10^3, x_hist[22:24,end].*10^3)
r2, v2 = kepler_to_rv(oe2.a, oe2.e, oe2.i, oe2.Ω, oe2.f+oe2.ω, 0)#oe0.f+oe0.ω)

oe3 = rv_to_kepler(x_hist[31:33,end].*10^3, x_hist[34:36,end].*10^3)
r3, v3 = kepler_to_rv(oe3.a, oe3.e, oe3.i, oe3.Ω, oe3.f+oe3.ω, 0)#oe1.f+oe1.ω)


# angle = asin(norm(cross(r0,r1))/(norm(r0)*norm(r1)))
# println(rad2deg(angle))
# distance = norm(r1)*angle
# println(distance)
rv_to_kepler(r2,v2)

In [ ]:
using JLD2
temp = (Δθhist, Δahist, ΔΩhist, u1hist, u2hist, a1hist, a2hist, θ1hist, θ2hist, Ω1hist, Ω2hist, tp_hist, Δθunhist, x_hist, t_hist, params)
if save_res ==1
    save_object("Data/4sat_Jeffects_rot_eccen_control1_Box.jld2", temp)
end


In [ ]:
println(r0)
# pl= scatter([r0[1]],[r0[2]], [r0[3]]),


# x = [r0[1]/10,2,3]
# r0 = r0.*10^-3
pl = scatter([tuple(r0./(Re_mks*10^3)...)], label = "Sat. 1", markersize = 2)
scatter!([tuple(r1./(Re_mks*10^3)...)], label = "Sat. 2", markersize = 2)
scatter!([tuple(r2./(Re_mks*10^3)...)], label = "Sat. 3", markersize = 2)
scatter!([tuple(r3./(Re_mks*10^3)...)], label = "Sat. 4", markersize = 2)

function sphere(r, C)   # r: radius; C: center [cx,cy,cz]
    n = 100
    u = range(-π, π; length = n)
    v = range(0, π; length = n)
    x = C[1] .+ r*cos.(u) * sin.(v)'
    y = C[2] .+ r*sin.(u) * sin.(v)'
    z = C[3] .+ r*ones(n) * cos.(v)'
    return x, y, z
end

println(min(r0[1],r1[1],r2[1],r3[1]), (max(r0[1],r1[1],r2[1],r3[1])), (min(r0[2],r1[2],r2[2],r3[2])), (max(r0[2],r1[2],r2[2],r3[2])))

x,y,z = sphere(1, [0,0,0])

# println(round(atand(1 / √2); digits = 3))
kw = (aspect_ratio = :equal, label = :none, size=(500,500), xlabel = "x", ylabel = "y", zlabel = "z", xlim = [-0.2,1.2], )#proj_type = :orthozlim = [-1.,1.], xlim = [-1.2,1.2], ylim = [-1.2,1.2],
plot!(x, y, z; color=:blue, alpha=:0.1, camera = (90+90, 100+210), kw...) #130



#, plot(x, y, z; proj_type = :persp, title = "perspective (fps)", camera = (0, 0), kw...))


savefig(pl, "Plots/try.pdf")

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))
model_scale = 10

vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 

# (2*(0.2*0.15) + (0.1*0.15))*10^-6/1.504

size_spacecraft = Vec(0.2*10^-3,0.2*10^-3,0.2*10^-3)
size_plate = Vec(size_spacecraft[1]*3,0.005,0.2*10^-3)

origin_spacecraft = Vec(x_hist[1,1]/Re_mks-size_spacecraft[1]/2,x_hist[2,1]/Re_mks-size_spacecraft[2]/2,x_hist[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x_hist[1,1]/Re_mks-size_plate[1]/2,x_hist[2,1]/Re_mks-size_plate[2]/2,x_hist[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)


colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

# setprop!(vis["Earth"], "scale", model_scale * ones(3))
# setprop!(vis["Earth"], "scale", model_scale * ones(3))

positions_0 = [r0[1]/Re_mks-origin_spacecraft[1],r0[1]/Re_mks-origin_spacecraft[2],r0[2]/Re_mks-origin_spacecraft[3]]
position_0 = Translation(positions_0)
name_sat = "satellite1"
settransform!(vis[name_sat], position_0)

positions_1 = [r1[1]/Re_mks-origin_spacecraft[1],r1[1]/Re_mks-origin_spacecraft[2],r1[2]/Re_mks-origin_spacecraft[3]]
position_1 = Translation(positions_1)
name_sat = "satellite2"
settransform!(vis[name_sat], position_1)

positions_2 = [r2[1]/Re_mks-origin_spacecraft[1],r2[1]/Re_mks-origin_spacecraft[2],r2[2]/Re_mks-origin_spacecraft[3]]
position_2 = Translation(positions_2)
name_sat = "satellite3"
settransform!(vis[name_sat], position_2)

positions_3 = [r3[1]/Re_mks-origin_spacecraft[1],r3[1]/Re_mks-origin_spacecraft[2],r3[2]/Re_mks-origin_spacecraft[3]]
position_3 = Translation(positions_3)
name_sat = "satellite4"
settransform!(vis[name_sat], position_3)

# for j = 0 : num_sat-1
#     if j == 0
#         positions_1 = [x_hist[1,end]/Re_mks-origin_spacecraft[1],x_hist[2,end]/Re_mks-origin_spacecraft[2],x_hist[3,end]/Re_mks-origin_spacecraft[3]]
#         position_1 = Translation(positions_1)
#         name_sat = "satellite"*string(j+1)
#         settransform!(vis[name_sat], position_1)
#     end 
#     positions_2 = [x_hist[7+(12*j),end]/Re_mks-origin_spacecraft[1],x_hist[8+(12*j),end]/Re_mks-origin_spacecraft[2],x_hist[9+(12*j),end]/Re_mks-origin_spacecraft[3]]
#     position_2 = Translation(positions_2)
#     name_sat = "satellite"*string(j+2)
#     settransform!(vis[name_sat], position_2)
# end



In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x_hist[1,1]/Re_mks-size_spacecraft[1]/2,x_hist[2,1]/Re_mks-size_spacecraft[2]/2,x_hist[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x_hist[1,1]/Re_mks-size_plate[1]/2,x_hist[2,1]/Re_mks-size_plate[2]/2,x_hist[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

sleep(5)
for i = 1:10:length(x_hist[1,:])-1
    for j = 0 : num_sat-1
        if j == 0
#             positions_1 = [x0_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            positions_1 = [x_hist[1,i]/Re_mks-origin_spacecraft[1],x_hist[2,i]/Re_mks-origin_spacecraft[2],x_hist[3,i]/Re_mks-origin_spacecraft[3]]
#             rotations_1 = i*2*pi/length(x0_soln[1+(6*j-6),:])
            position_1 = Translation(positions_1)
#             rotation_1 = LinearMap(RotZ(rotations_1))
            name_sat = "satellite"*string(j+1)
            settransform!(vis[name_sat], position_1)
#             settransform!(vis[name_sat], compose(position_1, rotation_1))
        end 
#         positions_2 = [x1_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
        positions_2 = [x_hist[7+(12*j),i]/Re_mks-origin_spacecraft[1],x_hist[8+(12*j),i]/Re_mks-origin_spacecraft[2],x_hist[9+(12*j),i]/Re_mks-origin_spacecraft[3]]
#         rotations_2 = i*2*pi/length(x1_soln[1+(6*j-6),:])
        position_2 = Translation(positions_2)
#         rotation_2 = LinearMap(RotZ(rotations_2))
        name_sat = "satellite"*string(j+2)
        settransform!(vis[name_sat], position_2)
#         settransform!(vis[name_sat], compose(position_2, rotation_2))
    end
        #     settransform!(vis["plate1"], position)
    sleep(0.05)
end


In [ ]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    r_noJ2 = x[7:9]
    v_noJ2 = x[10:12]

    pos_x = r[1]
    pos_y = r[2]
    pos_z = r[3]
    r_norm = sqrt(r[1]^2+r[2]^2+r[3]^2)
    r_norm_noJ2 = sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2)
    
    g = +1.5*J2*μ_mks*Re_mks^2/r_norm^4*[
        pos_x/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_y/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_z/r_norm*(5*pos_z^2/r_norm^2-3)] - μ_mks*r/r_norm^3
    
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = density_mks(r_norm - Re_mks)
    Fdrag = -0.5*(u*AoM_mks)*Cd*ρ*(sqrt(v_rel[1]^2+v_rel[2]^2+v_rel[3]^2))*v_rel

    v_rel_noJ2 = v_noJ2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ_noJ2 = density_mks(r_norm_noJ2 - Re_mks)
    Fdrag_noJ2 = -0.5*(u*AoM_mks)*Cd*ρ_noJ2*(sqrt(v_rel_noJ2[1]^2+v_rel_noJ2[2]^2+v_rel_noJ2[3]^2))*v_rel_noJ2

    v̇ = g + Fdrag
    v̇_noJ2 = -μ_mks/(r_norm_noJ2)^3 * r_noJ2 +  Fdrag_noJ2
    return [v; v̇; v_noJ2; v̇_noJ2]
end


In [ ]:
#Initial conditions in Cartesian Coordinates
# r0 = [a; 0; 0]
# v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
# x0 = [r0; v0; r0; v0];

# r1 = [a; 0; 0]
# v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
# x1 = [r1; v1; r1; v1];

# tspan = (0.0, Tfinal+50)
# Nt_t = Tfinal+150

r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

r1 = [a_mks; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ_mks/(a_mks)); sind(inc)*sqrt(μ_mks/(a_mks))]
x1 = [r1; v1; r1; v1];

step_t = 60*10 # save results every 10 mins
tspan = (0.0, Int(round((Tfinal)*P_mks/step_t)*step_t))
Nt_t = Int(round((Tfinal)*P_mks/step_t))-1#/100

In [ ]:
using Interpolations
#Full Orbit Mechanics Simulation and save results
# Nt = 150
# Nt = Nt*3
temp = LinRange(tspan[1],tspan[2],Nt_t)
x0_soln = zeros(6*num_sat,Nt_t)
x1_soln = zeros(6*num_sat,Nt_t)

x0_noJ2_soln = zeros(6*num_sat,Nt_t)
x1_noJ2_soln = zeros(6*num_sat,Nt_t)

Δa_soln = zeros(num_sat,Nt_t)
Δθ_soln = zeros(num_sat,Nt_t)
ΔΩ_soln = zeros(num_sat,Nt_t)

r_soln1 = zeros(num_sat,Nt_t)
ϕ_soln1 = zeros(num_sat,Nt_t)
θ_soln1 = zeros(num_sat,Nt_t)
u_soln1 = zeros(num_sat,Nt_t)

r_soln2 = zeros(num_sat,Nt_t)
ϕ_soln2 = zeros(num_sat,Nt_t)
θ_soln2 = zeros(num_sat,Nt_t)
u_soln2 = zeros(num_sat,Nt_t)

a_soln1 = zeros(num_sat,Nt_t)
ν_soln1 = zeros(num_sat,Nt_t)
Ω_soln1 = zeros(num_sat,Nt_t)

a_soln2 = zeros(num_sat,Nt_t)
ν_soln2 = zeros(num_sat,Nt_t)
Ω_soln2 = zeros(num_sat,Nt_t)

r_soln1_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln1_noJ2 = zeros(num_sat,Nt_t)
θ_soln1_noJ2 = zeros(num_sat,Nt_t)
u_soln1_noJ2 = zeros(num_sat,Nt_t)

r_soln2_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln2_noJ2 = zeros(num_sat,Nt_t)
θ_soln2_noJ2 = zeros(num_sat,Nt_t)
u_soln2_noJ2 = zeros(num_sat,Nt_t)

h_traj = 1#step_t#/P_mks
thist_c = thist.*P_mks
for i = 1:num_sat
    u1interp = 0;#LinearInterpolation(thist_c, u1hist[:,i])
    u2interp = 1;#LinearInterpolation(thist_c, u2hist[:,i])

    x_hist1 = zeros(length(x0),Int(tspan[2]/h_traj))
    x_hist2 = zeros(length(x1),Int(tspan[2]/h_traj))
    x_hist1[:,1] = x0 
    x_hist2[:,1] = x1 
    temp = LinRange(tspan[1],tspan[2],Nt_t)
    k = 0 
    save_index = 0
    for t = tspan[1]:h_traj:tspan[2]-step_t# temp# 1:100:tspan[2]-1
        k += 1
        #Stored Controller
        if t < tspan[2]
            u1 = 0;#u1interp(t) 
            u2 = 1;#u2interp(t) 
        else
            u1 = 0;#u_soln1[i,k]
            u2 = 0;#u_soln2[i,k]
        end

        x_hist1[:,k+1] .= sat_dynamics_rk4(x_hist1[:,k],u1,h_traj)
        x_hist2[:,k+1] .= sat_dynamics_rk4(x_hist2[:,k],u2,h_traj)

        # save results
        if k%step_t == 0
            save_index += 1
            u_soln1[i,save_index] = u1
            u_soln2[i,save_index] = u2
            x0_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[1:6,k]
            x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[7:12,k]
        
            x1_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[1:6,k]
            x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[7:12,k]
    
            rϕθ1_noJ2 = cartesian_to_spherical(x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2_noJ2 = cartesian_to_spherical(x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ1 = cartesian_to_spherical(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2 = cartesian_to_spherical(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            ΔaθΩ = cartesian_to_ΔθaΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]), mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ1 = cartesian_to_θaΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ2 = cartesian_to_θaΩ(mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
    
            Δθ_soln[i,save_index] = ΔaθΩ[1]    
            Δa_soln[i,save_index] = ΔaθΩ[2]
            ΔΩ_soln[i,save_index] = ΔaθΩ[3]

            ν_soln1[i,save_index] = aθΩ1[1]
            a_soln1[i,save_index] = aθΩ1[2]
            Ω_soln1[i,save_index] = aθΩ1[3]
        
            ν_soln2[i,save_index] = aθΩ2[1]
            a_soln2[i,save_index] = aθΩ2[2]
            Ω_soln2[i,save_index] = aθΩ2[3]
        
            r_soln1_noJ2[i,save_index] = rϕθ1_noJ2[1]
            ϕ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[2]
            θ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[3]
    
            r_soln2_noJ2[i,save_index] = rϕθ2_noJ2[1]
            ϕ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[2]
            θ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[3]
        
            r_soln1[i,save_index] = rϕθ1[1]
            ϕ_soln1[i,save_index] = rϕθ1[2]
            θ_soln1[i,save_index] = rϕθ1[3]
    
            r_soln2[i,save_index] = rϕθ2[1]
            ϕ_soln2[i,save_index] = rϕθ2[2]
            θ_soln2[i,save_index] = rϕθ2[3]
        end

    end
end
tsamp = LinRange(tspan[1],tspan[2],Nt_t);


In [ ]:
using Plots
# tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp[1:30:end]./(60*60*24),x0_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[3,1:30:end],color=:green)

plot!(tsamp[1:30:end]./(60*60*24),x1_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[3,1:30:end],color=:green)

# plot(tsamp,x0_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x0_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x0_noJ2_soln[3,:],color=:green)

# plot!(tsamp,x1_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x1_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x1_noJ2_soln[3,:],color=:green)
# x1_soln[3,:]

In [ ]:
plot(tsamp[1:30:end]./(60*60*24),broadcast.(rad2deg,ν_soln1[1,1:30:end]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln2[1,:]),color=:blue)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

# length(Ω_soln1[1,:][Ω_soln1[1,:] .>3.])

In [ ]:
tsamp = LinRange(tspan[1],tspan[2],Nt_t)
plot(tsamp[1:30:end]./(60*60*24),Δa_soln[1:30:end]',label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "Altitude Difference, km")


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Δθ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "True Anomaly Difference, deg")

In [ ]:
plot(tsamp./(60*40*24),broadcast.(rad2deg,ΔΩ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "RAAN Difference, deg", ylim = [-10,5])

In [ ]:
plot(tsamp,a_mks*ΔΩ_soln',label=permutedims(mylabel), ylim = [-100*1e1,10*1e2])
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
plot(tsamp./(60*60*24),u_soln1',color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),u_soln2',label=permutedims(mylabel2))
plot!(xlabel="Time, days", ylabel = "Control")


In [ ]:
plot(tsamp./(60*40*24),r_soln1_noJ2'.-Re_mks,color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*40*24),r_soln2_noJ2'.-Re_mks,label=permutedims(mylabel2))
for i=1:num_sat
    println(r_soln1_noJ2[1]-Re_mks," ",params.Δa_max*a_mks*i)
    plot!([tsamp[1]./(60*40*24), tsamp[end]./(60*40*24)],[r_soln1_noJ2[1]-Re_mks-params.Δa_max*a_mks*i, r_soln1_noJ2[1]-Re_mks[1]-params.Δa_max*a_mks*i],color=:red, linewidth=2,label="Limit")
end
plot!(xlabel= "Time, days",ylabel= "Altitude, km")


In [ ]:
plot(tsamp[1:30:end]./(60*40*24),ϕ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),ϕ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Azimuth, rad")

In [ ]:
plot(tsamp[1:30:end]./(60*40*24),θ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),θ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Elevation, rad")

In [ ]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [ ]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [ ]:
k1/k2

In [ ]:
(k1/k2)*2*pi

In [ ]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi